# 🧵 Day 02a: Threads — Lightweight Processes

---

## 🎯 What You'll Master Today
- Thread vs process — what's shared, what's separate
- User-level vs kernel-level threads
- Multithreading models (many-to-one, one-to-one, many-to-many)
- Python threading, the GIL, and when to use threads vs processes

---

## 🎭 The Analogy

**Process** = separate kitchens with their own ingredients, stoves, and staff  
**Threads** = multiple chefs in the SAME kitchen sharing ingredients and tools  

Threads are lighter (shared resources), but dangerous (they can bump into each other — race conditions).

---

```
╔═══════════════════════════════════════════════════════════╗
║              THREAD vs PROCESS                            ║
╠════════════════════╦══════════════════════════════════════╣
║  Aspect             ║  Thread vs Process                  ║
╠════════════════════╬══════════════════════════════════════╣
║  Memory             ║  Thread: shares heap & data         ║
║                     ║  Process: separate memory space     ║
║  Creation cost      ║  Thread: lightweight (fast)         ║
║                     ║  Process: heavyweight (slow)        ║
║  Communication      ║  Thread: shared variables (fast)    ║
║                     ║  Process: IPC needed (slow)         ║
║  Crash impact       ║  Thread: kills entire process       ║
║                     ║  Process: isolated (safe)           ║
║  Own resources      ║  Thread: stack, PC, registers       ║
║                     ║  Process: everything                ║
╚════════════════════╩══════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. Basic Threading in Python
# ============================================
import threading
import time

def download_file(filename: str, size_mb: int):
    print(f"  ⬇️ Downloading {filename} ({size_mb}MB)... [Thread: {threading.current_thread().name}]")
    time.sleep(0.5)  # simulate download
    print(f"  ✅ {filename} complete!")

# Sequential — slow
print("Sequential:")
start = time.perf_counter()
download_file("video.mp4", 100)
download_file("image.png", 5)
download_file("data.csv", 50)
print(f"  Time: {time.perf_counter() - start:.2f}s\n")

# Threaded — fast (I/O operations overlap)
print("Threaded:")
start = time.perf_counter()
threads = [
    threading.Thread(target=download_file, args=("video.mp4", 100)),
    threading.Thread(target=download_file, args=("image.png", 5)),
    threading.Thread(target=download_file, args=("data.csv", 50)),
]
for t in threads: t.start()
for t in threads: t.join()
print(f"  Time: {time.perf_counter() - start:.2f}s")

In [ ]:
# ============================================
# 2. Shared State — Race Condition Demo
# ============================================
import threading

counter = 0

def unsafe_increment(n: int):
    global counter
    for _ in range(n):
        counter += 1  # NOT atomic! Read-modify-write

# Without lock — race condition
counter = 0
threads = [threading.Thread(target=unsafe_increment, args=(100_000,)) for _ in range(5)]
for t in threads: t.start()
for t in threads: t.join()
print(f"Expected: 500,000 | Got: {counter:,}  {'❌ RACE CONDITION!' if counter != 500_000 else '✅'}")

# With lock — safe
counter = 0
lock = threading.Lock()

def safe_increment(n: int):
    global counter
    for _ in range(n):
        with lock:  # acquire → critical section → release
            counter += 1

threads = [threading.Thread(target=safe_increment, args=(100_000,)) for _ in range(5)]
for t in threads: t.start()
for t in threads: t.join()
print(f"Expected: 500,000 | Got: {counter:,}  ✅ SAFE")

In [ ]:
# ============================================
# 3. The GIL — Python's Global Interpreter Lock
# ============================================
import threading
import multiprocessing
import time

def cpu_bound_task(n):
    """Heavy computation — GIL blocks parallelism."""
    total = 0
    for i in range(n):
        total += i * i
    return total

N = 5_000_000

# Threads — GIL prevents true parallelism for CPU-bound
start = time.perf_counter()
threads = [threading.Thread(target=cpu_bound_task, args=(N,)) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()
thread_time = time.perf_counter() - start

# Processes — bypass GIL, true parallelism
start = time.perf_counter()
processes = [multiprocessing.Process(target=cpu_bound_task, args=(N,)) for _ in range(4)]
for p in processes: p.start()
for p in processes: p.join()
process_time = time.perf_counter() - start

print(f"CPU-bound task × 4:")
print(f"  Threads:   {thread_time:.2f}s (GIL = no real parallelism)")
print(f"  Processes: {process_time:.2f}s (true parallelism)")

```
╔═══════════════════════════════════════════════════════════╗
║              WHEN TO USE WHAT                             ║
╠═══════════════════════════════════════════════════════════╣
║                                                           ║
║  I/O-bound (network, disk, DB):                           ║
║    → Use THREADS (threading) or ASYNC (asyncio)           ║
║    → GIL released during I/O wait                         ║
║                                                           ║
║  CPU-bound (math, image processing, ML):                  ║
║    → Use PROCESSES (multiprocessing)                      ║
║    → Bypass GIL with separate memory spaces               ║
║                                                           ║
║  Mixed:                                                   ║
║    → Use ProcessPoolExecutor + threads inside each        ║
║                                                           ║
╚═══════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | Thread vs Process? | Thread shares memory within process. Process has separate memory. Thread is lighter |
| 2 | What is the GIL? | Global Interpreter Lock — only 1 thread executes Python bytecode at a time. Affects CPU-bound, not I/O |
| 3 | User vs Kernel threads? | User threads managed by library (fast switch, but one blocks all). Kernel threads managed by OS (true parallelism) |
| 4 | Many-to-one vs one-to-one? | Many:1 = many user threads → 1 kernel (no parallelism). 1:1 = each user thread → 1 kernel (Linux uses this) |
| 5 | When threads vs processes in Python? | I/O-bound → threads. CPU-bound → processes. Always prefer asyncio for high-concurrency I/O |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Thread = lightweight, shares heap, own stack/PC       │
## │  • GIL = CPython bottleneck for CPU-bound threads        │
## │  • I/O-bound → threads. CPU-bound → processes            │
## │  • Shared state → race conditions → use locks            │
## │  • Linux uses 1:1 threading model                        │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **CPU Scheduling** — FCFS, SJF, Round Robin, and more